# Phase 2, Step 4b: Landsat Additional Bands Feature Extraction

This notebook extracts the three Landsat bands that were **not** collected in `Landsat_Data_Extraction_Notebook.ipynb`: **red**, **blue**, and **lwir11** (thermal infrared). It then computes derived spectral indices that require these bands and saves the results to new CSV files consumed by `05_Combine_All_Features.ipynb`.

## Why these bands?

| Band | Wavelength | Water quality relevance |
|------|-----------|-------------------------|
| `red` | 0.64–0.67 µm | Turbidity, sediment load, true NDVI |
| `blue` | 0.45–0.51 µm | Water clarity, EVI atmospheric correction |
| `lwir11` | 10.6–11.2 µm | Surface/water temperature proxy |

## Existing pipeline context

- `Landsat_Data_Extraction_Notebook.ipynb` extracted: `nir`, `green`, `swir16`, `swir22`, `NDMI`, `MNDWI`
- `02_Feature_Engineering_Spectral_Indices.ipynb` added 18 more derived indices (all from those 4 bands)
- This notebook fills the gap with bands requiring direct satellite extraction

## Outputs

- `landsat_extra_bands_training.csv` — 9,319 rows, 13 new feature columns
- `landsat_extra_bands_validation.csv` — 200 rows, same columns

`05_Combine_All_Features.ipynb` has been updated to load and merge these files automatically.

## Step 1: Load Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

from tqdm import tqdm
import os

tqdm.pandas()

In [ ]:
# Load sample locations from source datasets
print('Loading sample locations...')
water_quality_df = pd.read_csv('water_quality_training_dataset.csv')
validation_df    = pd.read_csv('submission_template.csv')

print(f'Training samples : {len(water_quality_df)}')
print(f'Validation samples: {len(validation_df)}')
water_quality_df.head()

## Step 2: Extract Red, Blue, and LWIR11 from Landsat

The extraction strategy mirrors `Landsat_Data_Extraction_Notebook.ipynb` exactly:

- **Bounding box**: ~100 m around each point (`bbox_size = 0.00089831°`)
- **Scene selection**: Landsat-8 C2 L2, cloud cover < 10%, pick the scene **closest to the sample date**
- **Aggregation**: Median pixel value within the bounding box
- **`lwir11`**: Kept as raw DN (no thermal scaling applied), consistent with how other bands are stored in the pipeline

The search window covers the full study period (2011–2015) to maximise the chance of finding a usable scene per point.

In [ ]:
def compute_additional_landsat_bands(row):
    """
    Extract red, blue, and lwir11 (thermal) Landsat-8 C2 L2 band values for a
    single sample point.

    Uses the same 100 m focal buffer and nearest-low-cloud-scene approach as
    Landsat_Data_Extraction_Notebook.ipynb.

    Parameters
    ----------
    row : pd.Series
        Must contain 'Latitude', 'Longitude', 'Sample Date' (DD-MM-YYYY).

    Returns
    -------
    pd.Series with keys: red, blue, lwir11
    """
    lat  = row['Latitude']
    lon  = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # ~100 m buffer in degrees (same as extraction notebook)
    bbox_size = 0.00089831
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2,
    ]

    nan_result = pd.Series({'red': np.nan, 'blue': np.nan, 'lwir11': np.nan})

    if pd.isna(date):
        return nan_result

    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=pc.sign_inplace,
    )

    search = catalog.search(
        collections=['landsat-c2-l2'],
        bbox=bbox,
        datetime='2011-01-01/2015-12-31',
        query={'eo:cloud_cover': {'lt': 10}},
    )
    items = search.item_collection()

    if not items:
        return nan_result

    try:
        sample_date_utc = (
            date.tz_localize('UTC') if date.tzinfo is None else date.tz_convert('UTC')
        )

        # Select the scene closest in time to the sample date
        items_sorted = sorted(
            items,
            key=lambda x: abs(
                pd.to_datetime(x.properties['datetime']).tz_convert('UTC')
                - sample_date_utc
            ),
        )
        selected_item = pc.sign(items_sorted[0])

        bands_of_interest = ['red', 'blue', 'lwir11']
        data = stac_load(
            [selected_item], bands=bands_of_interest, bbox=bbox
        ).isel(time=0)

        def safe_median(arr):
            val = float(arr.astype('float').median(skipna=True).values)
            return np.nan if val == 0 else val

        return pd.Series({
            'red'    : safe_median(data['red']),
            'blue'   : safe_median(data['blue']),
            'lwir11' : safe_median(data['lwir11']),
        })

    except Exception:
        return nan_result

### Extract for training dataset (9,319 points)

> **Note**: Like the original extraction notebook, this cell takes several hours to run on a typical laptop. Run in batches if needed and concatenate results before saving.

In [ ]:
TRAIN_RAW_PATH = 'landsat_extra_bands_raw_training.csv'

print('Extracting red / blue / lwir11 for training data...')
train_extra_raw = water_quality_df.progress_apply(compute_additional_landsat_bands, axis=1)

# Attach merge keys
train_extra_raw['Latitude']    = water_quality_df['Latitude'].values
train_extra_raw['Longitude']   = water_quality_df['Longitude'].values
train_extra_raw['Sample Date'] = water_quality_df['Sample Date'].values

# Reorder so merge keys come first
train_extra_raw = train_extra_raw[['Latitude', 'Longitude', 'Sample Date', 'red', 'blue', 'lwir11']]
train_extra_raw.to_csv(TRAIN_RAW_PATH, index=False)

print(f'Saved raw bands to {TRAIN_RAW_PATH} ({len(train_extra_raw)} rows)')
print(f'Missing values:\n{train_extra_raw[["red","blue","lwir11"]].isna().sum()}')
train_extra_raw.head()

### Extract for validation dataset (200 points)

In [ ]:
VAL_RAW_PATH = 'landsat_extra_bands_raw_validation.csv'

print('Extracting red / blue / lwir11 for validation data...')
val_extra_raw = validation_df.progress_apply(compute_additional_landsat_bands, axis=1)

val_extra_raw['Latitude']    = validation_df['Latitude'].values
val_extra_raw['Longitude']   = validation_df['Longitude'].values
val_extra_raw['Sample Date'] = validation_df['Sample Date'].values

val_extra_raw = val_extra_raw[['Latitude', 'Longitude', 'Sample Date', 'red', 'blue', 'lwir11']]
val_extra_raw.to_csv(VAL_RAW_PATH, index=False)

print(f'Saved raw bands to {VAL_RAW_PATH} ({len(val_extra_raw)} rows)')
print(f'Missing values:\n{val_extra_raw[["red","blue","lwir11"]].isna().sum()}')
val_extra_raw.head()

## Step 3: Compute Additional Spectral Indices

The new bands unlock indices that could not be computed from the original 4 bands (`nir`, `green`, `swir16`, `swir22`). Cross-band indices that also need `nir`, `green`, `swir16`, or `swir22` (e.g. AWEI) pull those values from the existing `landsat_features_training_enhanced.csv`.

### Index catalogue

| Index | Formula | Relevance |
|-------|---------|----------|
| `NDVI_red` | `(nir - red) / (nir + red)` | True NDVI; vegetation cover affects runoff and water quality |
| `EVI` | `2.5*(nir_r - red_r) / (nir_r + 6*red_r - 7.5*blue_r + 1)` | Enhanced vegetation; reduces soil/atmospheric bias vs NDVI |
| `SAVI` | `(nir - red) / (nir + red + 0.5) * 1.5` | Soil-adjusted vegetation; better in semi-arid Southern Africa |
| `BSI_full` | `((swir16 + red) - (nir + blue)) / ((swir16 + red) + (nir + blue))` | Full bare soil index; erosion drives sediment in water |
| `AWEI_nsh` | `4*(green - swir16) - (0.25*nir + 2.75*swir22)` | Water extraction, no shadow version |
| `AWEI_sh` | `blue + 2.5*green - 1.5*(nir + swir16) - 0.25*swir22` | Water extraction, shadow-aware version |
| `NRI` | `(green - red) / (green + red)` | Normalised Red Index; plant stress / sediment |
| `Red_Blue_ratio` | `red / blue` | Water clarity proxy; high ratio = turbid/coloured water |
| `Green_Red_ratio` | `green / red` | Chlorophyll / algae signal |
| `Turbidity_Red` | `red / green` | Red-band turbidity; suspended sediment |

> **EVI scaling**: EVI requires physical reflectance (0–1). Raw Landsat C2 L2 DNs are converted inline with the standard formula: `reflectance = DN * 0.0000275 - 0.2`. The other indices are normalised ratios, so scaling cancels out.

In [ ]:
def compute_additional_indices(df_new, df_existing, epsilon=1e-10):
    """
    Compute spectral indices that require the newly extracted red, blue, lwir11
    bands, plus cross-band indices that also need nir/green/swir16/swir22 from
    the existing enhanced feature set.

    Parameters
    ----------
    df_new : pd.DataFrame
        Contains merge keys + 'red', 'blue', 'lwir11'.
    df_existing : pd.DataFrame
        Contains merge keys + 'nir', 'green', 'swir16', 'swir22'.

    Returns
    -------
    pd.DataFrame
        Merge-key columns + raw new bands + all new derived indices.
    """
    MERGE_KEYS = ['Latitude', 'Longitude', 'Sample Date']

    # Normalise merge keys: round coords, standardise date format
    def norm(df):
        out = df.copy()
        dates = pd.to_datetime(out['Sample Date'], dayfirst=True, errors='coerce')
        out['Sample Date'] = dates.dt.strftime('%d-%m-%Y').where(dates.notna(), out['Sample Date'])
        out['Latitude']  = out['Latitude'].round(6)
        out['Longitude'] = out['Longitude'].round(6)
        return out

    df_new      = norm(df_new)
    df_existing = norm(df_existing)

    # Merge new bands with the existing nir/green/swir bands for cross-band indices
    existing_cols = MERGE_KEYS + ['nir', 'green', 'swir16', 'swir22']
    merged = df_new.merge(
        df_existing[existing_cols], on=MERGE_KEYS, how='left'
    )

    red    = merged['red'].values.astype(float)
    blue   = merged['blue'].values.astype(float)
    nir    = merged['nir'].values.astype(float)
    green  = merged['green'].values.astype(float)
    swir16 = merged['swir16'].values.astype(float)
    swir22 = merged['swir22'].values.astype(float)

    out = merged[MERGE_KEYS + ['red', 'blue', 'lwir11']].copy()

    # ── True NDVI (red band) ─────────────────────────────────────────────────
    out['NDVI_red'] = np.clip((nir - red) / (nir + red + epsilon), -1, 1)

    # ── EVI (requires physical reflectance 0-1) ──────────────────────────────
    SCALE, OFFSET = 0.0000275, -0.2
    nir_r  = nir  * SCALE + OFFSET
    red_r  = red  * SCALE + OFFSET
    blue_r = blue * SCALE + OFFSET
    evi_denom = nir_r + 6.0 * red_r - 7.5 * blue_r + 1.0
    out['EVI'] = np.where(
        np.abs(evi_denom) < epsilon,
        np.nan,
        np.clip(2.5 * (nir_r - red_r) / evi_denom, -1, 1),
    )

    # ── SAVI (L = 0.5) ──────────────────────────────────────────────────────
    out['SAVI'] = np.clip(
        (nir - red) / (nir + red + 0.5 + epsilon) * 1.5, -1.5, 1.5
    )

    # ── BSI full (uses all four original bands + red + blue) ─────────────────
    bsi_num = (swir16 + red) - (nir + blue)
    bsi_den = (swir16 + red) + (nir + blue)
    out['BSI_full'] = np.clip(bsi_num / (bsi_den + epsilon), -1, 1)

    # ── AWEI no-shadow ───────────────────────────────────────────────────────
    out['AWEI_nsh'] = 4.0 * (green - swir16) - (0.25 * nir + 2.75 * swir22)

    # ── AWEI shadow-aware ────────────────────────────────────────────────────
    out['AWEI_sh'] = blue + 2.5 * green - 1.5 * (nir + swir16) - 0.25 * swir22

    # ── NRI ─────────────────────────────────────────────────────────────────
    out['NRI'] = np.clip((green - red) / (green + red + epsilon), -1, 1)

    # ── Band ratios ──────────────────────────────────────────────────────────
    out['Red_Blue_ratio']  = red   / (blue  + epsilon)
    out['Green_Red_ratio'] = green / (red   + epsilon)
    out['Turbidity_Red']   = red   / (green + epsilon)

    return out

In [ ]:
# Load existing enhanced features for cross-band index computation
print('Loading existing enhanced Landsat features...')
existing_train = pd.read_csv('landsat_features_training_enhanced.csv')
existing_val   = pd.read_csv('landsat_features_validation_enhanced.csv')
print(f'Existing training shape : {existing_train.shape}')
print(f'Existing validation shape: {existing_val.shape}')

In [ ]:
# Compute new indices for training
print('Computing additional indices for training data...')
train_extra = compute_additional_indices(train_extra_raw, existing_train)

new_index_cols = [
    'NDVI_red', 'EVI', 'SAVI', 'BSI_full',
    'AWEI_nsh', 'AWEI_sh', 'NRI',
    'Red_Blue_ratio', 'Green_Red_ratio', 'Turbidity_Red',
]
print(f'Shape: {train_extra.shape}')
print(f'New index columns: {new_index_cols}')
train_extra.head()

In [ ]:
# Compute new indices for validation
print('Computing additional indices for validation data...')
val_extra = compute_additional_indices(val_extra_raw, existing_val)

print(f'Shape: {val_extra.shape}')
val_extra.head()

In [ ]:
# Summary statistics for new indices (training)
feature_cols = [c for c in train_extra.columns if c not in ['Latitude', 'Longitude', 'Sample Date']]
print('Summary statistics for new features (training):')
print(train_extra[feature_cols].describe().round(4))

print('\nMissing value counts (training):')
print(train_extra[feature_cols].isna().sum())

## Step 4: Save New Feature CSVs

The output files are saved as **new** CSVs. The existing `landsat_features_training_enhanced.csv` and `landsat_features_validation_enhanced.csv` are **not modified**. `05_Combine_All_Features.ipynb` loads these files as an additional feature source and merges them by `(Latitude, Longitude, Sample Date)`.

In [ ]:
OUT_TRAIN = 'landsat_extra_bands_training.csv'
OUT_VAL   = 'landsat_extra_bands_validation.csv'

train_extra.to_csv(OUT_TRAIN, index=False)
val_extra.to_csv(OUT_VAL, index=False)

print(f'Saved {OUT_TRAIN} ({len(train_extra)} rows, {len(train_extra.columns)} columns)')
print(f'Saved {OUT_VAL}   ({len(val_extra)} rows, {len(val_extra.columns)} columns)')
print(f'\nColumns: {list(train_extra.columns)}')

## Summary

### New raw bands extracted

| Band | Description | Raw units |
|------|-------------|----------|
| `red` | Visible red (Band 4, 0.64–0.67 µm) | DN (same scale as nir/green/swir in pipeline) |
| `blue` | Visible blue (Band 2, 0.45–0.51 µm) | DN |
| `lwir11` | Thermal infrared (Band 10, 10.6–11.2 µm) | DN |

### New derived indices

| Index | Formula key | Notes |
|-------|------------|-------|
| `NDVI_red` | `(nir - red) / (nir + red)` | True NDVI; contrast with non-standard NDVI in enhanced CSV |
| `EVI` | `2.5*(nir_r - red_r) / (nir_r + 6*red_r - 7.5*blue_r + 1)` | Scaled to physical reflectance |
| `SAVI` | `(nir - red) / (nir + red + 0.5) * 1.5` | L=0.5 standard |
| `BSI_full` | `((swir16+red)-(nir+blue)) / ((swir16+red)+(nir+blue))` | Full 4-band bare soil index |
| `AWEI_nsh` | `4*(green-swir16) - (0.25*nir + 2.75*swir22)` | Open water, no shadows |
| `AWEI_sh` | `blue + 2.5*green - 1.5*(nir+swir16) - 0.25*swir22` | Open water, shadow-aware |
| `NRI` | `(green - red) / (green + red)` | Plant stress / sediment |
| `Red_Blue_ratio` | `red / blue` | Water colour / clarity |
| `Green_Red_ratio` | `green / red` | Chlorophyll signal |
| `Turbidity_Red` | `red / green` | Suspended sediment proxy |

### Next step

Run `05_Combine_All_Features.ipynb` — it has been updated to load `landsat_extra_bands_training.csv` and `landsat_extra_bands_validation.csv` and merge them into the final combined feature dataset.